# 뉴스 백필 — 키워드 추출 + 감성분석 (GPU 서버)

Google Drive의 백필 뉴스 CSV에서 키워드 추출 + 감성분석을 수행합니다.

## 구성
- **키워드 추출 + LLM 감성**: vLLM (Qwen2.5-7B-Instruct) — 한 번의 호출로 동시 처리
- **감성분석 (기준)**: FinBERT (snunlp/KR-FinBert-SC)
- **하드 샘플**: FinBERT vs LLM 불일치 → 파인튜닝용 JSONL 내보내기

## 사전 준비
```bash
pip install vllm transformers torch pandas tqdm gdown openai
# vLLM 서버 실행 (별도 터미널)
vllm serve Qwen/Qwen2.5-7B-Instruct --dtype auto --max-model-len 4096 --gpu-memory-utilization 0.5
```

## Drive 폴더 구조
```
📁 GDRIVE_FOLDER_ID/
├── 2025-20260316/    ← 최신부터 처리
├── 2024/
├── 2023/
├── 2022/
├── 2021/
├── 2020/
└── 2019/
    └── {종목코드}_{종목명}.csv
```

## 0. 패키지 설치 + Google Drive 다운로드

In [ ]:
!pip install -q openai transformers torch pandas tqdm

In [ ]:
import zipfile
from pathlib import Path

# ============================================================
# 데이터 경로 설정
# JupyterHub에 업로드한 zip 파일을 자동으로 해제합니다.
# 구조: DATA_ROOT/{연도}/{종목코드}_{종목명}.csv
# ============================================================
DATA_ROOT = Path("./data/news_backfill")
ZIP_DIR = Path("./data/zips")  # zip 파일 업로드 경로

DATA_ROOT.mkdir(parents=True, exist_ok=True)

# zip 파일 자동 해제
if ZIP_DIR.exists():
    for zip_path in sorted(ZIP_DIR.glob("*.zip")):
        # 손상된 zip 스킵
        if not zipfile.is_zipfile(zip_path):
            size_mb = zip_path.stat().st_size / 1024 / 1024
            print(f"⚠️ {zip_path.name} ({size_mb:.1f}MB) — 유효하지 않은 zip, 스킵. 다시 다운로드하세요.")
            continue

        # 폴더명: 2025_20260316.zip → 2025-20260316
        folder_name = zip_path.stem.replace("_", "-")
        output_dir = DATA_ROOT / folder_name

        if output_dir.exists() and any(output_dir.iterdir()):
            print(f"⏭️ {zip_path.name} → 이미 해제됨, 스킵")
            continue

        print(f"📦 압축 해제 중: {zip_path.name} → {output_dir}")
        output_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(output_dir)
        print(f"✅ 완료: {zip_path.name}")

# 폴더 존재 확인
if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
    years = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
    total_csvs = sum(1 for _ in DATA_ROOT.rglob("*.csv"))
    print(f"\n✅ 데이터 경로: {DATA_ROOT}")
    print(f"   연도 폴더: {years}")
    print(f"   전체 CSV: {total_csvs}개")
else:
    print(f"❌ 데이터가 없습니다.")
    print(f"   Drive에서 연도 폴더별로 zip 다운로드 후")
    print(f"   {ZIP_DIR}/ 에 업로드하세요.")

## 1. 설정 + 공통 import

In [ ]:
import os
import json
import re
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm.notebook import tqdm

# WARNING 이상만 출력 (건별 INFO 로그 방지)
logging.basicConfig(level=logging.WARNING, format="%(asctime)s | %(levelname)-7s | %(message)s")
logger = logging.getLogger(__name__)

# ============================================================
# 설정값
# ============================================================

# vLLM 서버 주소 (별도 터미널에서 vllm serve 실행 필요)
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# FinBERT 모델
FINBERT_MODEL = "snunlp/KR-FinBert-SC"

# 처리 결과 저장 경로
OUTPUT_ROOT = Path("./output/backfill_processed")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 하드 샘플 저장 경로
HARD_SAMPLE_DIR = Path("./output/hard_samples")
HARD_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# 배치 크기
LLM_BATCH_SIZE = 50        # vLLM 동시 요청 수
FINBERT_BATCH_SIZE = 64    # FinBERT 배치 크기

# 하드 샘플 판정 기준
CONFIDENCE_THRESHOLD = 0.7

# 처리 순서: 최신 연도부터 (역순)
YEAR_ORDER = [
    "2025-20260316",
    "2024",
    "2023",
    "2022",
    "2021",
    "2020",
    "2019",
]

## 2. vLLM 서버 연결 확인

In [ ]:
from openai import OpenAI

# vLLM은 OpenAI-compatible API를 제공
llm_client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")

# 서버 연결 테스트
try:
    models = llm_client.models.list()
    print(f"vLLM 서버 연결 성공. 사용 가능 모델: {[m.id for m in models.data]}")
except Exception as e:
    print(f"vLLM 서버 연결 실패: {e}")
    print("\nvLLM 서버를 먼저 실행하세요:")
    print(f"  vllm serve {VLLM_MODEL} --dtype auto --max-model-len 4096 --gpu-memory-utilization 0.5")

## 3. FinBERT 로드

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LABEL_MAP = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}

print(f"FinBERT 모델 로딩: {FINBERT_MODEL}")
finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
finbert_device = "cuda" if torch.cuda.is_available() else "cpu"
finbert_model.to(finbert_device)
finbert_model.eval()
print(f"FinBERT 로딩 완료 (device={finbert_device})")

## 4. 핵심 함수 정의

In [ ]:
# ---------------------------------------------------------------------------
# vLLM: 키워드 + 감성 동시 추출
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = (
    "당신은 한국 주식 시장 뉴스 분석 전문가입니다. "
    "주어진 기사에서 핵심 키워드를 추출하고 감성을 분류합니다."
)

USER_PROMPT_TEMPLATE = """다음 주식 뉴스를 분석해주세요.

1. 핵심 주제 키워드를 최대 3개 추출 (산업 트렌드/이슈/현상 명사구, 2~8글자, 기업명 제외)
2. 감성 분석: POSITIVE / NEGATIVE / NEUTRAL 중 하나 + 확신도 (0.0~1.0)

반드시 아래 JSON 형식으로만 응답하세요. 다른 텍스트는 쓰지 마세요.
{{"keywords": ["키워드1", "키워드2"], "sentiment_label": "POSITIVE", "sentiment_score": 0.85}}

제목: {title}
본문: {body}

JSON:"""


def parse_llm_response(raw: str) -> dict:
    """LLM 응답에서 JSON 파싱. 실패 시 기본값 반환."""
    try:
        # JSON 객체 추출
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            result = json.loads(match.group())
            keywords = result.get("keywords", [])
            keywords = [str(k).strip() for k in keywords[:5] if k and str(k).strip()]
            # 8글자 초과 키워드 필터
            keywords = [k for k in keywords if len(k) <= 20]

            label = result.get("sentiment_label", "NEUTRAL").upper()
            if label not in ("POSITIVE", "NEGATIVE", "NEUTRAL"):
                label = "NEUTRAL"
            score = float(result.get("sentiment_score", 0.5))
            score = max(0.0, min(1.0, score))

            return {"keywords": keywords, "sentiment_label": label, "sentiment_score": score}
    except (json.JSONDecodeError, ValueError, KeyError):
        pass
    return {"keywords": [], "sentiment_label": "NEUTRAL", "sentiment_score": 0.0}


def call_vllm_batch(articles: list[dict]) -> list[dict]:
    """
    vLLM 서버에 배치 요청. 각 기사에 대해 키워드 + 감성을 추출.
    반환: [{"keywords": [...], "sentiment_label": "...", "sentiment_score": ...}, ...]
    """
    results = []
    for article in articles:
        title = str(article.get("title", ""))
        body = str(article.get("body", ""))[:1500]
        prompt = USER_PROMPT_TEMPLATE.format(title=title, body=body)

        try:
            response = llm_client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                max_tokens=150,
                temperature=0.1,
            )
            raw = response.choices[0].message.content.strip()
            results.append(parse_llm_response(raw))
        except Exception as e:
            logger.warning("vLLM 호출 실패: %s", e)
            results.append({"keywords": [], "sentiment_label": "NEUTRAL", "sentiment_score": 0.0})

    return results

In [ ]:
# ---------------------------------------------------------------------------
# vLLM 비동기 배치 (throughput 향상)
# ---------------------------------------------------------------------------
import asyncio
from openai import AsyncOpenAI

async_llm_client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key="not-needed")


async def _call_vllm_single(title: str, body: str, semaphore: asyncio.Semaphore) -> dict:
    """단일 기사에 대한 비동기 vLLM 호출."""
    prompt = USER_PROMPT_TEMPLATE.format(title=title, body=body[:1500])
    async with semaphore:
        try:
            response = await async_llm_client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                max_tokens=150,
                temperature=0.1,
            )
            raw = response.choices[0].message.content.strip()
            return parse_llm_response(raw)
        except Exception as e:
            return {"keywords": [], "sentiment_label": "NEUTRAL", "sentiment_score": 0.0}


async def call_vllm_batch_async(articles: list[dict], max_concurrent: int = 50) -> list[dict]:
    """
    비동기로 vLLM 배치 요청. vLLM의 continuous batching을 최대한 활용.
    max_concurrent: 동시 요청 수 제한 (vLLM이 내부적으로 배치 처리)
    """
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [
        _call_vllm_single(
            str(a.get("title", "")),
            str(a.get("body", "")),
            semaphore,
        )
        for a in articles
    ]
    return await asyncio.gather(*tasks)

In [ ]:
# ---------------------------------------------------------------------------
# FinBERT 배치 감성분석
# ---------------------------------------------------------------------------
def run_finbert_batch(texts: list[str], batch_size: int = 64) -> list[dict]:
    """
    FinBERT 배치 감성분석.
    반환: [{"label": "POSITIVE", "score": 0.92}, ...]
    """
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = finbert_tokenizer(
            batch_texts, return_tensors="pt", truncation=True,
            max_length=512, padding=True,
        )
        inputs = {k: v.to(finbert_device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = finbert_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)

        for j in range(len(batch_texts)):
            pred_idx = probs[j].argmax().item()
            results.append({
                "label": LABEL_MAP[pred_idx],
                "score": round(probs[j][pred_idx].item(), 4),
            })

    return results

In [ ]:
# ---------------------------------------------------------------------------
# 하드 샘플 판정
# ---------------------------------------------------------------------------
def detect_hard_samples(
    df: pd.DataFrame,
    threshold: float = 0.7,
) -> pd.DataFrame:
    """
    FinBERT vs LLM 감성분석 결과를 비교하여 hard sample 플래그를 추가.
    
    Hard sample 조건 (OR):
      - 라벨 불일치 (finbert_label != llm_label)
      - FinBERT 저신뢰 (finbert_score < threshold)
      - LLM 저신뢰 (llm_score < threshold)
    """
    df["is_label_mismatch"] = df["finbert_label"] != df["llm_label"]
    df["is_low_conf_finbert"] = df["finbert_score"] < threshold
    df["is_low_conf_llm"] = df["llm_score"] < threshold
    df["is_hard_sample"] = (
        df["is_label_mismatch"] | df["is_low_conf_finbert"] | df["is_low_conf_llm"]
    )
    return df

## 5. CSV 파일 목록 수집 (최신 연도부터)

In [ ]:
def collect_csv_files(data_root: Path, year_order: list[str]) -> list[Path]:
    """
    Drive 폴더에서 처리할 CSV 파일 목록을 수집.
    year_order 순서대로 (최신 연도부터) 정렬.
    """
    csv_files = []

    for year_name in year_order:
        year_dir = data_root / year_name
        if not year_dir.is_dir():
            logger.warning("폴더 없음 (스킵): %s", year_dir)
            continue
        for csv_file in sorted(year_dir.glob("*.csv")):
            csv_files.append(csv_file)

    return csv_files


csv_files = collect_csv_files(DATA_ROOT, YEAR_ORDER)
print(f"처리 대상 CSV 파일: {len(csv_files)}개\n")

# 연도별 분포 (처리 순서대로)
year_counts = {}
for f in csv_files:
    year = f.parent.name
    year_counts[year] = year_counts.get(year, 0) + 1

for year in YEAR_ORDER:
    if year in year_counts:
        print(f"  {year}: {year_counts[year]}개 종목")

## 6. 체크포인트 관리

In [ ]:
CHECKPOINT_PATH = OUTPUT_ROOT / "_checkpoint.json"


def load_checkpoint() -> set[str]:
    """이미 처리 완료된 CSV 파일 경로 set 반환."""
    if CHECKPOINT_PATH.is_file():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def save_checkpoint(done: set[str]) -> None:
    """처리 완료된 파일 목록 저장."""
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(sorted(done), f, ensure_ascii=False, indent=2)


done_files = load_checkpoint()
pending_files = [f for f in csv_files if str(f) not in done_files]
print(f"처리 완료: {len(done_files)}개, 남은 파일: {len(pending_files)}개")

## 7. 단일 CSV 처리 함수

In [ ]:
async def process_single_csv(csv_path: Path) -> dict:
    """
    단일 CSV 파일을 처리:
      1. CSV 로드 (title, article_url, source, date, code, name, body, full_body)
      2. vLLM으로 키워드 + LLM 감성 추출 (비동기 배치)
      3. FinBERT로 감성분석 (배치)
      4. 하드 샘플 판정
      5. 결과 CSV 저장
    
    반환: {"file": str, "total": int, "hard_samples": int}
    """
    # 1. CSV 로드
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype={"code": str}, engine="python", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(csv_path, encoding="utf-8", dtype={"code": str}, engine="python", on_bad_lines="skip")

    if df.empty:
        return {"file": csv_path.name, "total": 0, "hard_samples": 0}

    # code 컬럼 패딩
    if "code" in df.columns:
        df["code"] = df["code"].astype(str).str.zfill(6)

    # 입력 텍스트 준비
    articles = []
    for _, row in df.iterrows():
        articles.append({
            "title": str(row.get("title", "")),
            "body": str(row.get("full_body", row.get("body", "")))[:1500],
        })

    # 2. vLLM: 키워드 + LLM 감성 (비동기 배치)
    llm_results = await call_vllm_batch_async(articles, max_concurrent=LLM_BATCH_SIZE)

    df["keywords"] = [",".join(r["keywords"]) for r in llm_results]
    df["llm_label"] = [r["sentiment_label"] for r in llm_results]
    df["llm_score"] = [r["sentiment_score"] for r in llm_results]

    # 3. FinBERT 감성분석
    finbert_texts = [
        f"{row.get('title', '')} {str(row.get('body', ''))[:250]}"
        for _, row in df.iterrows()
    ]
    finbert_results = run_finbert_batch(finbert_texts, batch_size=FINBERT_BATCH_SIZE)

    df["finbert_label"] = [r["label"] for r in finbert_results]
    df["finbert_score"] = [r["score"] for r in finbert_results]

    # 4. 하드 샘플 판정
    df = detect_hard_samples(df, threshold=CONFIDENCE_THRESHOLD)

    # 5. 최종 감성 결정: FinBERT 기준 (primary)
    # sentiment_score를 -1.0 ~ 1.0 범위로 변환
    def to_sentiment_score(row):
        if row["finbert_label"] == "POSITIVE":
            return round(row["finbert_score"], 4)
        elif row["finbert_label"] == "NEGATIVE":
            return round(-row["finbert_score"], 4)
        return 0.0

    df["sentiment_score"] = df.apply(to_sentiment_score, axis=1)
    df["sentiment_label"] = df["finbert_label"]

    # 6. 결과 CSV 저장 (연도 폴더 구조 유지)
    year = csv_path.parent.name
    out_dir = OUTPUT_ROOT / year
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / csv_path.name

    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    hard_count = df["is_hard_sample"].sum()
    return {"file": csv_path.name, "total": len(df), "hard_samples": int(hard_count)}

## 8. 전체 배치 실행

In [ ]:
async def run_backfill():
    """전체 백필 CSV를 순차 처리 (파일 단위 체크포인트)."""
    done_files = load_checkpoint()
    pending = [f for f in csv_files if str(f) not in done_files]

    if not pending:
        print("처리할 파일이 없습니다.")
        return

    print(f"\n{'='*60}")
    print(f"  백필 처리 시작: {len(pending)}개 파일")
    print(f"{'='*60}\n")

    total_articles = 0
    total_hard = 0
    all_hard_samples = []

    for idx, csv_path in enumerate(tqdm(pending, desc="CSV 처리"), 1):
        try:
            result = await process_single_csv(csv_path)

            total_articles += result["total"]
            total_hard += result["hard_samples"]

            # 체크포인트 갱신
            done_files.add(str(csv_path))
            save_checkpoint(done_files)

            if idx % 20 == 0:
                logger.info(
                    "진행: %d/%d | 기사 %d건 | 하드샘플 %d건",
                    idx, len(pending), total_articles, total_hard,
                )

        except Exception as e:
            logger.error("[%s] 처리 실패: %s", csv_path.name, e)
            continue

    print(f"\n{'='*60}")
    print(f"  백필 완료")
    print(f"  처리 파일: {len(pending)}개")
    print(f"  전체 기사: {total_articles}건")
    print(f"  하드 샘플: {total_hard}건 ({total_hard/max(total_articles,1)*100:.1f}%)")
    print(f"  결과 경로: {OUTPUT_ROOT}")
    print(f"{'='*60}")


# 실행
await run_backfill()

## 9. 하드 샘플 내보내기

In [ ]:
def export_all_hard_samples():
    """
    처리 완료된 CSV에서 하드 샘플만 추출하여 JSONL로 내보내기.
    파인튜닝 데이터셋 구축용.
    """
    hard_records = []

    for csv_path in sorted(OUTPUT_ROOT.rglob("*.csv")):
        if csv_path.name.startswith("_"):
            continue
        try:
            df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype={"code": str})
        except Exception:
            continue

        if "is_hard_sample" not in df.columns:
            continue

        hard = df[df["is_hard_sample"] == True]
        for _, row in hard.iterrows():
            hard_records.append({
                "title": row.get("title", ""),
                "body": str(row.get("body", ""))[:500],
                "code": row.get("code", ""),
                "date": row.get("date", ""),
                "finbert_label": row.get("finbert_label", ""),
                "finbert_score": row.get("finbert_score", 0),
                "llm_label": row.get("llm_label", ""),
                "llm_score": row.get("llm_score", 0),
                "is_label_mismatch": bool(row.get("is_label_mismatch", False)),
                "is_low_conf_finbert": bool(row.get("is_low_conf_finbert", False)),
                "is_low_conf_llm": bool(row.get("is_low_conf_llm", False)),
            })

    # JSONL 저장
    today_str = datetime.now().strftime("%Y%m%d")
    output_path = HARD_SAMPLE_DIR / f"hard_samples_{today_str}.jsonl"

    with open(output_path, "w", encoding="utf-8") as f:
        for record in hard_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"하드 샘플 내보내기 완료: {output_path}")
    print(f"  전체: {len(hard_records)}건")

    # 통계
    if hard_records:
        mismatch = sum(1 for r in hard_records if r["is_label_mismatch"])
        low_fb = sum(1 for r in hard_records if r["is_low_conf_finbert"])
        low_llm = sum(1 for r in hard_records if r["is_low_conf_llm"])
        print(f"  라벨 불일치: {mismatch}건")
        print(f"  FinBERT 저신뢰: {low_fb}건")
        print(f"  LLM 저신뢰: {low_llm}건")

    return output_path


export_all_hard_samples()

## 10. 결과 통계 확인

In [ ]:
def print_summary():
    """처리된 전체 CSV의 통계를 출력."""
    total_articles = 0
    total_hard = 0
    label_dist = {"POSITIVE": 0, "NEGATIVE": 0, "NEUTRAL": 0}
    keyword_count = 0
    empty_keywords = 0

    for csv_path in sorted(OUTPUT_ROOT.rglob("*.csv")):
        if csv_path.name.startswith("_"):
            continue
        try:
            df = pd.read_csv(csv_path, encoding="utf-8-sig")
        except Exception:
            continue

        total_articles += len(df)

        if "is_hard_sample" in df.columns:
            total_hard += df["is_hard_sample"].sum()

        if "sentiment_label" in df.columns:
            for label in ["POSITIVE", "NEGATIVE", "NEUTRAL"]:
                label_dist[label] += (df["sentiment_label"] == label).sum()

        if "keywords" in df.columns:
            empty_keywords += (df["keywords"].fillna("").str.strip() == "").sum()
            keyword_count += df["keywords"].fillna("").str.count(",").sum() + len(df)

    print(f"\n{'='*60}")
    print(f"  전체 결과 통계")
    print(f"{'='*60}")
    print(f"  전체 기사: {total_articles:,}건")
    print(f"  하드 샘플: {total_hard:,.0f}건 ({total_hard/max(total_articles,1)*100:.1f}%)")
    print(f"  키워드 추출 실패: {empty_keywords:,}건")
    print(f"\n  감성 분포:")
    for label, count in label_dist.items():
        pct = count / max(total_articles, 1) * 100
        print(f"    {label}: {count:,}건 ({pct:.1f}%)")
    print(f"{'='*60}")


print_summary()

## 11. (참고) vLLM 서버 실행 명령어

```bash
# 기본 실행 (GPU 메모리 50% 사용 — FinBERT와 공유)
vllm serve Qwen/Qwen2.5-7B-Instruct \
    --dtype auto \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.5

# AWQ 양자화 모델 사용 시 (메모리 절약)
vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ \
    --dtype auto \
    --quantization awq \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.5

# 모델만 바꾸고 싶을 때 (14B)
vllm serve Qwen/Qwen2.5-14B-Instruct-AWQ \
    --dtype auto \
    --quantization awq \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.6
```